<a href="https://colab.research.google.com/github/SattamAltwaim/StarX/blob/main/experiments/15_assembly_synthetic_sketches.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Setup: clone the StarX repo and the pinned TripoSR commit, install this
# notebook's dependencies, mount Drive, and report what machine we are on.
import os
import subprocess
import sys

BRANCH = "main"
TRIPOSR_COMMIT = "107cefdc244c39106fa830359024f6a2f1c78871"
NOTEBOOK_ID = "15"

IN_COLAB = os.path.exists("/content")
if IN_COLAB:
    REPO_DIR = "/content/StarX"
    if not os.path.exists(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--branch", BRANCH,
             "https://github.com/SattamAltwaim/StarX.git", REPO_DIR],
            check=True,
        )
    TRIPOSR_DIR = "/content/TripoSR"
else:
    # off Colab the kernel's cwd is unpredictable (VS Code often starts in
    # $HOME) - walk up from the notebook location and the cwd to find the
    # repo, with ~/StarX as the final fallback
    def _find_repo():
        candidates = [globals().get("__vsc_ipynb_file__"), os.getcwd()]
        for start in candidates:
            if not start:
                continue
            path = os.path.abspath(
                os.path.dirname(start) if os.path.isfile(start) else start
            )
            while path != os.path.dirname(path):
                if os.path.exists(os.path.join(path, "starx", "pins.py")):
                    return path
                path = os.path.dirname(path)
        home_repo = os.path.join(os.path.expanduser("~"), "StarX")
        if os.path.exists(os.path.join(home_repo, "starx", "pins.py")):
            return home_repo
        raise RuntimeError(
            "could not locate the StarX repo - start Jupyter inside it "
            "or clone it to ~/StarX"
        )

    REPO_DIR = _find_repo()
    TRIPOSR_DIR = os.path.join(REPO_DIR, "third_party", "TripoSR")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

if not os.path.exists(TRIPOSR_DIR):
    subprocess.run(
        ["git", "clone", "https://github.com/VAST-AI-Research/TripoSR.git",
         TRIPOSR_DIR],
        check=True,
    )
subprocess.run(["git", "-C", TRIPOSR_DIR, "checkout", "-q", TRIPOSR_COMMIT], check=True)

from starx import pins

assert pins.TRIPOSR_COMMIT == TRIPOSR_COMMIT, "notebook pin out of sync with starx/pins.py"
# .get, not [NOTEBOOK_ID]: same situation as notebooks 13/14 - this pin
# never made it into the upstream starx/pins.py, so this must degrade to
# "nothing extra" instead of KeyError-ing.
if pins.PIP_PINS.get(NOTEBOOK_ID):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *pins.PIP_PINS[NOTEBOOK_ID]],
        check=True,
    )
# trimesh + pyrender for headless rendering, py7zr for the .7z archive -
# none of these are guaranteed by the upstream pins file for this notebook.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "trimesh", "pyrender", "py7zr"],
    check=True,
)

from starx import colab as scolab

DRIVE = scolab.mount_drive()
report = scolab.setup_report()

# 15 - Assembly bodies into 16 synthetic sketches

The pipeline you actually asked for: for each assembly body, render its
mesh from 16 posed cameras and Sobel-edge each render into a synthetic line
drawing - the same "one sketch in, many supervision views" input notebook
09/10/11 build for the reconstruction dataset via `starx.render_gt` +
`starx.cameras` + `starx.synth`. Nothing in those three modules is
reconstruction-specific; they operate on a normalized mesh and a design id,
not on which dataset it came from. Adapted here to read `.obj` bodies out
of `j1.0.0.7z` instead of the reconstruction zip.

The camera setup is the exact one notebook 03 uses and never tunes -
16 views, elevation -10 to 45 degrees, fixed camera distance/FOV/scene
radius baked into the pretrained TripoSR checkpoint - so sketches produced
here are directly compatible with `starx.train.paper_train_step`.

**Deliberately skipped: notebook 03's alignment gate.** That step loads the
actual pretrained TripoSR checkpoint (~1.7 GB) and proves pyrender's cameras
agree with TripoSR's own renderer before trusting hours of rendering. This
notebook validates the render + sketch pipeline on a sample and a small
batch, not a full production build - run that gate for real before treating
these renders as trustworthy supervision at dataset scale.

Everything this notebook writes lands on Drive, matching your earlier ask,
not the Colab session's local disk.

Use a GPU runtime if you have one - EGL rendering uses it; falls back to
OSMesa (CPU, slower but correct) automatically otherwise.

In [ ]:
# Configuration - every tunable for this notebook lives here.
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import py7zr
import trimesh
from PIL import Image
from tqdm.auto import tqdm

from starx import cameras, render_gt, synth, viz
from starx.config import CAMERA_DISTANCE, FOVY_DEG, SCENE_RADIUS, StarXConfig

SMOKE = True     # True: smaller batch for a fast pipeline check
SEED = 1337

ASSEMBLY_DATA_URL = (
    "https://fusion-360-gallery-dataset.s3.us-west-2.amazonaws.com/"
    "assembly/j1.0.0/j1.0.0.7z"
)

cfg = StarXConfig(
    drive_root=str(DRIVE / "StarX")
    if DRIVE is not None
    else os.path.join(REPO_DIR, "data", "StarX"),
    gt_size=256,
    n_views=16,
    elev_range=(-10.0, 45.0),
    seed=SEED,
)
archive_drive_path = Path(cfg.drive_root) / "raw" / "j1.0.0.7z"

# Everything lands under here, on Drive.
OUTPUT_DIR = Path(cfg.drive_root) / "assembly_sketch_check"
SAMPLE_DIR = OUTPUT_DIR / "sample"
BATCH_DIR = OUTPUT_DIR / "batch"
FIGURE_DIR = OUTPUT_DIR / "figures"
SKETCH_DIR = OUTPUT_DIR / "sketches"
for _d in (SAMPLE_DIR, BATCH_DIR, FIGURE_DIR, SKETCH_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# fixed by the pretrained checkpoint - printed, never tuned
print(f"camera distance {CAMERA_DISTANCE}   fovy {FOVY_DEG} deg   scene radius {SCENE_RADIUS}")
print(f"views per body: {cfg.n_views}   elevation range: {cfg.elev_range}")
print("archive on Drive lands at:", archive_drive_path)
print("sketches land at:", SKETCH_DIR)

In [ ]:
# Get the archive onto fast local disk (and archived on Drive). Same
# download logic as notebooks 13/14 - redefined here since it isn't in the
# upstream starx.colab (hardcoded to r1.0.1.zip / zipfile there).
def ensure_assembly_archive_local(url, drive_path, local_dir):
    local_archive = Path(local_dir) / "j1.0.0.7z"
    if local_archive.exists():
        return local_archive
    if drive_path.exists():
        return scolab.copy_with_progress(drive_path, local_archive)
    scolab.download_with_progress(url, local_archive)
    if Path(cfg.drive_root).exists():
        scolab.copy_with_progress(local_archive, drive_path)
    return local_archive


archive_local_dir = "/content" if IN_COLAB else os.path.join(REPO_DIR, "data")
archive_path = ensure_assembly_archive_local(
    ASSEMBLY_DATA_URL, archive_drive_path, archive_local_dir
)
size_gib = archive_path.stat().st_size / 2**30
print(f"archive ready at {archive_path} ({size_gib:.2f} GiB)")

In [ ]:
# Every per-body .obj under j1.0.0/joint/ is its own design, the same unit
# notebook 14 rasterizes and notebook 03 renders for the reconstruction
# dataset - confirmed by the file layout notebook 13 found.
with py7zr.SevenZipFile(archive_path, mode="r") as _archive:
    names = _archive.getnames()

obj_names = [
    n for n in names if n.startswith("j1.0.0/joint/") and n.endswith(".obj")
]
design_ids = sorted(os.path.splitext(os.path.basename(n))[0] for n in obj_names)
print(f"{len(design_ids)} bodies with a .obj under j1.0.0/joint/")

In [ ]:
# Start the offscreen renderer and prove it works - same class notebook 03
# uses, EGL (GPU) tried first, OSMesa (CPU) fallback.
renderer = render_gt.GTRenderer(cfg.gt_size)
print("renderer backend:", renderer.backend)

In [ ]:
# Pick one sample body, extract its mesh, normalize it into TripoSR's scene
# cube, sample 16 posed cameras (deterministic per body id, seeded - same
# convention notebook 03 uses), and render.
rng = random.Random(SEED)
SAMPLE_ID = rng.choice(design_ids)
sample_obj_member = f"j1.0.0/joint/{SAMPLE_ID}.obj"
with py7zr.SevenZipFile(archive_path, mode="r") as _archive:
    _archive.extract(path=SAMPLE_DIR, targets=[sample_obj_member])

raw_mesh = trimesh.load(SAMPLE_DIR / sample_obj_member, force="mesh")
norm_mesh, mesh_center, mesh_scale = render_gt.normalize_mesh(raw_mesh)
sample_c2ws, sample_angles = cameras.sample_design_views(
    SAMPLE_ID, cfg.n_views, cfg.elev_range, seed=cfg.seed
)
sample_rgbs, sample_masks = renderer.render_mesh(norm_mesh, sample_c2ws)
print(f"rendered {len(sample_rgbs)} views of {SAMPLE_ID}")

titles = [f"az {a:.0f} el {e:.0f}" for a, e in sample_angles]
fig = viz.show_view_grid(list(sample_rgbs), list(sample_masks), titles=titles)
fig.savefig(FIGURE_DIR / f"{SAMPLE_ID}_renders.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Sobel-edge every one of those 16 renders into a synthetic sketch - the
# same conversion notebook 09/10 use, via starx.synth (config-driven, so
# this and training-time can't drift apart on edge-detection parameters).
sample_sketches = synth.sketches_for_views(sample_rgbs, cfg, device="cpu")
sketch_imgs = (sample_sketches.numpy() * 255).astype(np.uint8)

n_cols = cfg.n_views // 2
fig, axes = plt.subplots(2, n_cols, figsize=(1.8 * n_cols, 4))
for ax, img, (az, el) in zip(axes.ravel(), sketch_imgs, sample_angles):
    ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"az {az:.0f} el {el:.0f}", fontsize=8)
    ax.axis("off")
fig.tight_layout()
fig.savefig(FIGURE_DIR / f"{SAMPLE_ID}_sketches.png", dpi=150, bbox_inches="tight")
plt.show()

# Save the 16 sketches plus the cameras that produced them for this body.
sample_body_dir = SKETCH_DIR / SAMPLE_ID
sample_body_dir.mkdir(parents=True, exist_ok=True)
for i, img in enumerate(sketch_imgs):
    Image.fromarray(img).save(sample_body_dir / f"sketch_{i:02d}.png")
np.save(sample_body_dir / "c2ws.npy", sample_c2ws)
np.save(sample_body_dir / "view_angles.npy", sample_angles)
print(f"saved {len(sketch_imgs)} sketches -> {sample_body_dir}")

In [ ]:
# Batch-test across a random sample of bodies: extract obj -> normalize ->
# sample 16 cameras -> render -> sketch -> save, catching failures per body
# instead of letting one bad mesh kill the run. Rendering is the slow part
# here (16 renders per body), so this defaults small under SMOKE - raise
# BATCH_N once this looks right.
BATCH_N = 10 if SMOKE else 100
batch_ids = rng.sample(design_ids, min(BATCH_N, len(design_ids)))

failures = []
for design_id in tqdm(batch_ids, desc="rendering+sketching"):
    try:
        member = f"j1.0.0/joint/{design_id}.obj"
        with py7zr.SevenZipFile(archive_path, mode="r") as _archive:
            _archive.extract(path=BATCH_DIR, targets=[member])
        mesh = trimesh.load(BATCH_DIR / member, force="mesh")
        norm, _, _ = render_gt.normalize_mesh(mesh)
        c2ws_b, angles_b = cameras.sample_design_views(
            design_id, cfg.n_views, cfg.elev_range, seed=cfg.seed
        )
        rgbs_b, _ = renderer.render_mesh(norm, c2ws_b)
        sketches_b = (
            synth.sketches_for_views(rgbs_b, cfg, device="cpu").numpy() * 255
        ).astype(np.uint8)

        body_dir = SKETCH_DIR / design_id
        body_dir.mkdir(parents=True, exist_ok=True)
        for i, img in enumerate(sketches_b):
            Image.fromarray(img).save(body_dir / f"sketch_{i:02d}.png")
        np.save(body_dir / "c2ws.npy", c2ws_b)
        np.save(body_dir / "view_angles.npy", angles_b)
    except Exception as error:
        failures.append({"design_id": design_id, "error": repr(error)})

print(f"clean: {len(batch_ids) - len(failures)}/{len(batch_ids)}")
pd.DataFrame(failures).head(10) if failures else print("no failures")

## Takeaways

- Every body that rendered cleanly now has 16 synthetic sketches on Drive
  at `assembly_sketch_check/sketches/<body_id>/sketch_00.png` ...
  `sketch_15.png`, plus `c2ws.npy` / `view_angles.npy` - the cameras that
  produced them. That's the same "one sketch in, the rest as supervision
  views" shape `starx.train.paper_train_step` expects.
- **Not done here on purpose:** notebook 03's alignment gate (proving
  pyrender agrees with TripoSR's own renderer) - do that with the real
  checkpoint before trusting these renders as actual training supervision,
  not just before a first look at them.
- **Still open:** the per-body vs per-joint-pair question from notebooks
  13/14. Every body renders independently here, same as a reconstruction
  design - whether a joint's two bodies should ever be supervised together
  is a modeling decision, not something this notebook resolves.
- This writes individual PNGs per body, not packed shards - fine for a
  sample and a small batch, but Drive is slow at many small files at real
  dataset scale. The notebook 03 equivalent (tar-shard packing, resumable)
  is the actual full build once this validates.